In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 83.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -q transformers datasets accelerate pandas openpyxl tqdm

In [ ]:
# ===== IMPORTS =====
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from google.colab import files

# ===== MODEL NAME =====
model_name = "climatebert/distilroberta-base-climate-detector"

# ===== DEVICE =====
device = 0 if torch.cuda.is_available() else -1

# ===== LOAD MODEL =====
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)



# ===== PIPELINE =====
clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-detector
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


In [ ]:
output_file = "climatebert_detector_results.xlsx"
df.to_excel(output_file, index=False)

print("Saved:", output_file)
files.download(output_file)

Saved: climatebert_detector_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Whole Code


In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from google.colab import files

# Upload file
uploaded = files.upload()

# Load Excel
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# Clean data
df = df.copy()
df["paragraph"] = df["paragraph"].fillna("").astype(str).str.strip()
df = df[df["paragraph"] != ""].reset_index(drop=True)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Load model
model_name = "climatebert/distilroberta-base-climate-detector"
device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Label mapping:", model.config.id2label)

# Run inference
texts = df["paragraph"].tolist()
all_results = []
batch_size = 32

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_results = clf(batch, truncation=True, padding=True, batch_size=batch_size)
    all_results.extend(batch_results)

df["detector_label"] = [r["label"] for r in all_results]
df["detector_score"] = [r["score"] for r in all_results]

# IMPORTANT: inspect label meaning before deciding which label = climate-related
print(model.config.id2label)
print(model.config.label2id)

# Example only — change if needed after checking mapping
# df["is_climate_related"] = (df["detector_label"] == "LABEL_1").astype(int)

# Save full results
output_file = "climatebert_detector_results.xlsx"
df.to_excel(output_file, index=False)

print("Saved:", output_file)
files.download(output_file)